# 02 — Data Cleaning & ETL Pipeline

This notebook mirrors `scripts/etl_pipeline.py` so that:
- the cleaning is reproducible from a script (CI / batch runs),
- the notebook tells the story step-by-step for the final report.

Output: `data/processed/amazon_products_clean.csv`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from scripts import etl_pipeline as etl

RAW = ROOT / "data" / "raw" / "amazon_products_sales_data_uncleaned.csv"
OUT = ROOT / "data" / "processed" / "amazon_products_clean.csv"
RAW.exists(), OUT

(True,
 PosixPath('/Users/aashishjha/Documents/Projects/DVA-capstone-2/data/processed/amazon_products_clean.csv'))

## Step 1 — Load raw

In [2]:
df = etl.load_raw(RAW)
df.shape

2026-04-27 22:42:57,106 | INFO | Loading raw dataset: /Users/aashishjha/Documents/Projects/DVA-capstone-2/data/raw/amazon_products_sales_data_uncleaned.csv


2026-04-27 22:42:57,340 | INFO | Raw shape: 42675 rows × 16 cols


(42675, 16)

## Step 2 — Standardise column names

In [3]:
df = etl.standardise_columns(df)
list(df.columns)

2026-04-27 22:42:57,347 | INFO | Columns after standardisation: ['title', 'rating', 'number_of_reviews', 'bought_in_last_month', 'current_discounted_price', 'price_on_variant', 'listed_price', 'is_best_seller', 'is_sponsored', 'is_couponed', 'buy_box_availability', 'delivery_details', 'sustainability_badges', 'image_url', 'product_url', 'collected_at']


['title',
 'rating',
 'number_of_reviews',
 'bought_in_last_month',
 'current_discounted_price',
 'price_on_variant',
 'listed_price',
 'is_best_seller',
 'is_sponsored',
 'is_couponed',
 'buy_box_availability',
 'delivery_details',
 'sustainability_badges',
 'image_url',
 'product_url',
 'collected_at']

## Step 3 — Coerce types

Parsers (defined in `scripts/etl_pipeline.py`):
- `parse_rating` — pulls float out of `"4.6 out of 5 stars"`.
- `parse_count_shorthand` — handles `"1,234"`, `"6K+"`, `"1.2M"`.
- `parse_currency` — strips `$` and commas.
- `parse_coupon_pct` — extracts `%` from coupon strings, returns `0` for `"No Coupon"`.

In [4]:
df = etl.coerce_types(df)
df[["rating", "number_of_reviews", "bought_in_last_month", "current_discounted_price", "listed_price",
    "is_best_seller", "is_sponsored", "has_coupon", "coupon_pct", "has_buy_box"]].head()

2026-04-27 22:42:57,606 | INFO | Coerced dtypes:
title                                  str
rating                             float64
number_of_reviews                  float64
bought_in_last_month               float64
current_discounted_price           float64
price_on_variant                       str
listed_price                       float64
is_best_seller                        bool
is_sponsored                          bool
is_couponed                            str
buy_box_availability                   str
delivery_details                       str
sustainability_badges                  str
image_url                              str
product_url                            str
collected_at                datetime64[us]
has_buy_box                           bool
coupon_pct                         float64
has_coupon                            bool
has_sustainability_badge              bool
dtype: object


,rating,number_of_reviews,bought_in_last_month,current_discounted_price,listed_price,is_best_seller,is_sponsored,has_coupon,coupon_pct,has_buy_box
0,4.6,375.0,300.0,89.68,159.00,False,True,True,15.0,True
1,4.3,2457.0,6000.0,9.99,15.99,False,True,False,0.0,True
2,4.6,3044.0,2000.0,314.00,349.00,False,True,False,0.0,True
3,4.6,35882.0,10000.0,NaN,NaN,True,False,False,0.0,False
4,4.8,28988.0,10000.0,NaN,NaN,False,False,False,0.0,False


## Step 4 — Drop unusable rows + dedupe by title

In [5]:
before = len(df)
df = etl.drop_unusable_rows(df)
print(f"{before:,} → {len(df):,} rows  ({(before-len(df))/before:.1%} dropped)")

2026-04-27 22:42:57,624 | INFO | Dropped 36661 unusable / duplicate rows → 6014 remain


42,675 → 6,014 rows  (85.9% dropped)


## Step 5 — Engineer features

In [6]:
df = etl.engineer_features(df)
df[["title", "brand", "selling_price", "listed_price", "discount_pct", "is_discounted",
    "price_tier", "rating", "rating_bucket", "log_reviews"]].head()

2026-04-27 22:42:57,635 | INFO | Dropped 43 rows missing rating


2026-04-27 22:42:57,647 | WARNING | Columns still containing NaN:
buy_box_availability      513
delivery_details           66
sustainability_badges    5437
product_url               119
dtype: int64


2026-04-27 22:42:57,647 | INFO | Final columns: ['title', 'rating', 'number_of_reviews', 'bought_in_last_month', 'current_discounted_price', 'price_on_variant', 'listed_price', 'is_best_seller', 'is_sponsored', 'is_couponed', 'buy_box_availability', 'delivery_details', 'sustainability_badges', 'image_url', 'product_url', 'collected_at', 'has_buy_box', 'coupon_pct', 'has_coupon', 'has_sustainability_badge', 'selling_price', 'discount_pct', 'is_discounted', 'brand', 'price_tier', 'rating_bucket', 'log_reviews', 'log_bought_last_month']


,title,brand,selling_price,listed_price,discount_pct,is_discounted,price_tier,rating,rating_bucket,log_reviews
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,Boya,89.68,159.00,43.60,True,Premium,4.6,Excellent,5.929589
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",Lisen,9.99,15.99,37.52,True,Budget,4.3,Good,7.807103
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",Dji,314.00,349.00,10.03,True,Luxury,4.6,Excellent,8.021256
3,Complete Protect: One plan covers all eligible...,Complete,16.99,16.99,0.00,False,Mid,4.0,Good,8.385032
4,Amazon Basics 48-Pack AA Alkaline High-Perform...,Amazon,14.99,14.99,0.00,False,Budget,4.7,Excellent,13.671177


## Step 6 — Validate against Capstone 2 rules

In [7]:
assert len(df) >= 5_000, f"Row count {len(df)} below the 5K minimum"
assert df.shape[1] >= 8, "Need at least 8 analytical columns"
print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} cols  ✅")
df.dtypes

Final shape: 5,971 rows × 28 cols  ✅


title                                  str
rating                             float64
number_of_reviews                  float64
bought_in_last_month               float64
current_discounted_price           float64
price_on_variant                       str
listed_price                       float64
is_best_seller                        bool
is_sponsored                          bool
is_couponed                            str
buy_box_availability                   str
delivery_details                       str
sustainability_badges                  str
image_url                              str
product_url                            str
collected_at                datetime64[us]
has_buy_box                           bool
coupon_pct                         float64
has_coupon                            bool
has_sustainability_badge              bool
selling_price                      float64
discount_pct                       float64
is_discounted                         bool
brand      

## Step 7 — Drop noise columns and write

In [8]:
df = etl.drop_noise_columns(df)
etl.write_processed(df, OUT)
print("Wrote:", OUT)
df.head()

2026-04-27 22:42:57,694 | INFO | Wrote cleaned dataset: /Users/aashishjha/Documents/Projects/DVA-capstone-2/data/processed/amazon_products_clean.csv (5971 rows × 21 cols)


Wrote: /Users/aashishjha/Documents/Projects/DVA-capstone-2/data/processed/amazon_products_clean.csv


,title,rating,number_of_reviews,bought_in_last_month,current_discounted_price,listed_price,is_best_seller,is_sponsored,collected_at,has_buy_box,...,has_coupon,has_sustainability_badge,selling_price,discount_pct,is_discounted,brand,price_tier,rating_bucket,log_reviews,log_bought_last_month
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6,375.0,300.0,89.68,159.00,False,True,2025-08-21 11:14:29,True,...,True,True,89.68,43.60,True,Boya,Premium,Excellent,5.929589,5.707110
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3,2457.0,6000.0,9.99,15.99,False,True,2025-08-21 11:14:29,True,...,False,False,9.99,37.52,True,Lisen,Budget,Good,7.807103,8.699681
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6,3044.0,2000.0,314.00,349.00,False,True,2025-08-21 11:14:29,True,...,False,False,314.00,10.03,True,Dji,Luxury,Excellent,8.021256,7.601402
3,Complete Protect: One plan covers all eligible...,4.0,4380.0,0.0,16.99,16.99,False,False,2025-08-21 11:14:29,False,...,True,False,16.99,0.00,False,Complete,Mid,Good,8.385032,0.000000
4,Amazon Basics 48-Pack AA Alkaline High-Perform...,4.7,865598.0,100000.0,14.99,14.99,True,False,2025-08-21 11:14:29,True,...,False,True,14.99,0.00,False,Amazon,Budget,Excellent,13.671177,11.512935


## Cleaning log
| # | Action | Rationale |
|---|---|---|
| 1 | Load `data/raw/...uncleaned.csv` | Capstone rule — never edit raw |
| 2 | snake_case columns | downstream stability |
| 3 | parse rating / counts / currency / coupon | unlock numeric ops |
| 4 | drop rows with no title or no price; dedupe by title | data quality |
| 5 | engineer `selling_price`, `discount_pct`, `brand`, `price_tier`, `rating_bucket`, `log_reviews` | analysis-ready |
| 6 | assert capstone minimums | guardrail |
| 7 | drop URL / image / unstructured noise | small, focused extract |